# Crear dataftames CSV

## Location and Time

In [20]:
# Llegir el CSV conservant els codis com a text.
from pathlib import Path
import pandas as pd

data_dir = Path('dades')
if not data_dir.is_dir():
    data_dir = Path('VI_Project1/dades')

incendis_gencat = pd.read_csv(
    data_dir / 'Incendis_forestals_a_Catalunya._Anys_2011-2024_20261116.csv',
    dtype='string',
)
# ID propi segons la fila original del CSV (no es un ID oficial).
incendis_gencat['idincident'] = range(1, len(incendis_gencat) + 1)
incendis_gencat.head()

,DATA INCENDI,CODI COMARCA,COMARCA,CODI MUNICIPI,TERMEMUNIC,HAARBRADES,HANOARBRAD,HANOFOREST,HAFORESTAL,idincident
0,11/01/2011,08,Baix Camp,431299,Riudoms,"0,00","0,20","0,00","0,20",1
1,13/01/2011,37,Terra Alta,430483,Corbera d'Ebre,"0,20","0,00","0,00","0,20",2
2,14/01/2011,02,Alt Empordà,170545,Colera,"0,01","0,00","0,00","0,01",3
3,16/01/2011,06,Anoia,081022,Igualada,"0,00","0,02","0,00","0,02",4
4,16/01/2011,22,Montsià,430629,Freginals,"0,00","0,00","0,00","0,00",5


### Short dataset study (before cleaning)
Check NaNs and blank strings, exact repeated source rows, dates, code formats, and numeric area values before creating `locationtimeGenCat`. Compare duplicates across all nine original CSV columns, excluding the generated `idincident`. European numbers such as `5.621,80` mean 5621.80 hectares.


In [21]:
source_columns = incendis_gencat.columns.drop('idincident').tolist()
source_data = incendis_gencat[source_columns]
blank = source_data.apply(lambda column: column.str.strip().eq('').fillna(False))
missing = source_data.isna() | blank
exact_duplicates = source_data.duplicated(keep='first')
parsed_dates = pd.to_datetime(source_data['DATA INCENDI'], format='%d/%m/%Y', errors='coerce')
invalid_municipality = ~source_data['CODI MUNICIPI'].str.fullmatch(r'[0-9]{6}').fillna(False)
invalid_comarca = ~source_data['CODI COMARCA'].str.fullmatch(r'[0-9]{2}').fillna(False)
area_columns = ['HAARBRADES', 'HANOARBRAD', 'HANOFOREST', 'HAFORESTAL']
# Les superfícies del CSV fan servir punts de milers i comes decimals.
area_values = pd.DataFrame(index=source_data.index)
for column in area_columns:
    values = source_data[column].str.replace('.', '', regex=False)
    area_values[column] = pd.to_numeric(values.str.replace(',', '.', regex=False), errors='coerce')

invalid_areas = area_values.isna() & ~missing[area_columns]
forest_components = area_values['HAARBRADES'] + area_values['HANOARBRAD']
forest_total_mismatch = (forest_components - area_values['HAFORESTAL']).abs().gt(0.011)
print(f'Source: {len(source_data):,} rows, {len(source_columns)} columns')
print('Missing/blank values by column:')
print(missing.sum().to_string())
print(f'Exact duplicate rows beyond first: {exact_duplicates.sum()}')
print(f'Invalid/missing dates: {parsed_dates.isna().sum()} | Date range: {parsed_dates.min().date()} to {parsed_dates.max().date()}')
print(f'Malformed/missing municipality codes: {invalid_municipality.sum()} | Comarca codes: {invalid_comarca.sum()}')
print(f'Malformed nonmissing area values: {invalid_areas.sum().sum()} | Negative area values: {area_values.lt(0).sum().sum()}')
print(f'Rows with all four areas equal to zero: {area_values.eq(0).fillna(False).all(axis=1).sum()}')
print(f'Forest totals inconsistent by more than 0.011 ha (where components exist): {forest_total_mismatch.sum()}')
print('Rows with missing/blank values:')
print(incendis_gencat.loc[missing.any(axis=1)].to_string(index=False))


Source: 7,545 rows, 9 columns
Missing/blank values by column:
DATA INCENDI     0
CODI COMARCA     0
COMARCA          0
CODI MUNICIPI    0
TERMEMUNIC       0
HAARBRADES       0
HANOARBRAD       1
HANOFOREST       0
HAFORESTAL       0
Exact duplicate rows beyond first: 97
Invalid/missing dates: 0 | Date range: 2011-01-11 to 2024-12-31
Malformed/missing municipality codes: 0 | Comarca codes: 0
Malformed nonmissing area values: 0 | Negative area values: 0
Rows with all four areas equal to zero: 1360
Forest totals inconsistent by more than 0.011 ha (where components exist): 0
Rows with missing/blank values:
DATA INCENDI CODI COMARCA         COMARCA CODI MUNICIPI        TERMEMUNIC HAARBRADES HANOARBRAD HANOFOREST HAFORESTAL  idincident
  04/10/2018           41 Vallès Oriental        081249 Mollet del Vallès       0,04       <NA>       0,00       0,04        4401


### Findings and cleaning decisions
- **7,545 source rows; one missing value:** `HANOARBRAD` for Mollet del Valles on 04/10/2018. Keep this row because its location and date are complete and area is not used in `locationtimeGenCat`. Leave the missing source area unfilled.
- **97 exact duplicates beyond the first:** remove these before joining locations, keeping the first row and its original generated `idincident`. This leaves **7,448 rows**; gaps in IDs are intentional. Without an official incident ID or time of day, identical records could represent separate fires; removing them is the chosen cleaning rule.
- Dates and municipality/comarca code formats are valid. After European number parsing, there are no malformed nonmissing areas or negative areas; available forest totals agree within rounding tolerance. **1,360 rows have all four areas equal to zero**: retain them because zero is not missing and may reflect rounding or a small incident.
- Compare duplicates across the full original record, not only date/location: different fires can occur in the same municipality on the same day. Keep distinct source records even if their projected location/date fields match.
- Preserve the original source dataframe. The final dataframe uses reference municipality coordinates, not exact fire coordinates. Parse dates with `%d/%m/%Y` into pandas datetime; the source has no time of day. Validate the cleaned row count, unique generated IDs, and completeness of final location/date fields.


In [22]:
# Deduplicate all original columns, excluding the generated ID.
fires = incendis_gencat.drop_duplicates(subset=source_columns, keep='first').copy()
assert len(fires) == len(incendis_gencat) - int(exact_duplicates.sum())
assert not fires.duplicated(subset=source_columns).any()
incomplete_ids = incendis_gencat.loc[missing.any(axis=1), 'idincident']
assert incomplete_ids.isin(fires['idincident']).all()
print(f'Removed {len(incendis_gencat) - len(fires)} exact duplicates; kept {len(fires):,} incidents.')


Removed 97 exact duplicates; kept 7,448 incidents.


In [23]:
# Eliminar el digit de control per obtenir el codi municipal de cinc digits.
municipality_ids = fires['CODI MUNICIPI'].str.strip()
assert municipality_ids.str.fullmatch(r'[0-9]{6}').fillna(False).all(), 'Review missing or malformed municipality IDs'
fires['codi_municipi'] = municipality_ids.str[:5]
fires[['CODI MUNICIPI', 'codi_municipi']].head()

,CODI MUNICIPI,codi_municipi
0,431299,43129
1,430483,43048
2,170545,17054
3,081022,08102
4,430629,43062


In [24]:
# Preparar els noms i les divisions territorials amb la taula local.
municipalities = pd.read_csv(
    data_dir / 'corrMpisComsAtps.csv', sep=';', skiprows=3, dtype='string',
    usecols=['Codi municipi', 'Nom municipi', 'Codi província', 'Nom província', 'Nom comarca'],
)
municipalities = municipalities.rename(columns={
    'Nom municipi': 'municipi', 'Codi província': 'idprovincia',
    'Nom província': 'provincia', 'Nom comarca': 'comarca',
})
municipalities['codi_municipi'] = municipalities['Codi municipi'].str.strip().str[:5]
municipalities = municipalities[['codi_municipi', 'idprovincia', 'municipi', 'provincia', 'comarca']]

# Llegir les coordenades municipals i convertir les comes decimals.
coords = pd.read_csv(data_dir / 'MUNICIPIOS.csv', sep=';', encoding='cp1252', dtype='string')
coords = coords.loc[coords['COD_PROV'].isin(['08', '17', '25', '43'])].copy()
coords['codi_municipi'] = coords['COD_INE'].str.strip().str[:5]
coords['latitud'] = pd.to_numeric(coords['LATITUD_ETRS89_REGCAN95'].str.replace(',', '.', regex=False), errors='coerce')
coords['longitud'] = pd.to_numeric(coords['LONGITUD_ETRS89_REGCAN95'].str.replace(',', '.', regex=False), errors='coerce')
coords = coords[['codi_municipi', 'latitud', 'longitud']]

In [25]:
# Unir per codi municipal sense perdre ni duplicar registres.
merged = (
    fires.merge(municipalities, on='codi_municipi', how='left', validate='many_to_one')
    .merge(coords, on='codi_municipi', how='left', validate='many_to_one')
)
assert len(merged) == len(fires), 'The joins changed the number of fire records'
assert merged['idprovincia'].eq(merged['codi_municipi'].str[:2]).fillna(False).all(), 'Review unmatched or conflicting province IDs'

merged['idprovincia'] = merged['idprovincia'].astype('Int64')
merged['idmunicipio'] = merged['codi_municipi'].str[2:5].astype('Int64')
# El CSV nomes conte la data. Convertir-la a datetime 
merged['data_incendi'] = pd.to_datetime(merged['DATA INCENDI'], format='%d/%m/%Y', errors='raise')
location_columns = [
    'idincident', 'data_incendi', 'idprovincia', 'idmunicipio',
    'municipi', 'provincia', 'latitud', 'longitud', 'comarca',
]
locationtimeGenCat = merged[location_columns].copy()
locationtimeGenCat.head()

,idincident,data_incendi,idprovincia,idmunicipio,municipi,provincia,latitud,longitud,comarca
0,1,2011-01-11,43,129,Riudoms,Tarragona,41.137728,1.050838,Baix Camp
1,2,2011-01-13,43,48,Corbera d'Ebre,Tarragona,41.077737,0.478135,Terra Alta
2,3,2011-01-14,17,54,Colera,Girona,42.404019,3.150471,Alt Empordà
3,4,2011-01-16,8,102,Igualada,Barcelona,41.578714,1.617676,Anoia
4,5,2011-01-16,43,62,Freginals,Tarragona,40.671684,0.519778,Montsià


In [26]:
# Validate the cleaned location/date result.
assert len(locationtimeGenCat) == len(fires), 'The joins changed the cleaned row count'
assert locationtimeGenCat['idincident'].is_unique
assert not locationtimeGenCat.isna().any().any(), 'Review missing location/date fields'
assert pd.api.types.is_datetime64_any_dtype(locationtimeGenCat['data_incendi'])
print(f'Source rows: {len(incendis_gencat):,} | Removed duplicates: {exact_duplicates.sum()} | Location and Time rows: {len(locationtimeGenCat):,}')
print('Missing values in the final dataframe:')
print(locationtimeGenCat.isna().sum())
locationtimeGenCat.info()


Source rows: 7,545 | Removed duplicates: 97 | Location and Time rows: 7,448
Missing values in the final dataframe:
idincident      0
data_incendi    0
idprovincia     0
idmunicipio     0
municipi        0
provincia       0
latitud         0
longitud        0
comarca         0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7448 entries, 0 to 7447
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   idincident    7448 non-null   int64         
 1   data_incendi  7448 non-null   datetime64[ns]
 2   idprovincia   7448 non-null   Int64         
 3   idmunicipio   7448 non-null   Int64         
 4   municipi      7448 non-null   string        
 5   provincia     7448 non-null   string        
 6   latitud       7448 non-null   Float64       
 7   longitud      7448 non-null   Float64       
 8   comarca       7448 non-null   string        
dtypes: Float64(2), Int64(2), datetime64[ns](1), int64

In [27]:
locationtimeGenCat = locationtimeGenCat.rename(columns={
    "idincident": "numeroparte",
    "data_incendi": "data",
    "idmunicipio": "codi_municipi"
})

locationtimeGenCat.to_csv("Clean data/Clean GenCat Data.csv")